In [1]:
import pickle
import bw2io, bw2data
import bw2calc
import lca_algebraic as agb
import wurst as ws
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib import ticker
from matplotlib import rcParams
plt.rcParams["font.family"] = "Times New Roman"

### what activities we wanna run the calculation? 

In [ ]:
pkl_file = "filtered_agri_N201.pkl"
# Step 2: Load the file
with open(pkl_file, "rb") as f:
    agri = pickle.load(f)


In [ ]:
bw2data.projects.set_current('iveo_v1')
bw2data.databases

In [ ]:
eidb = 	bw2data.Database("ecoinvent 3.9.1")
print(len(eidb))

### let's compare GWP100 vs. pGWP100

In [ ]:
gwp_all = [
     ('IPCC 2021', 'climate change', 'GWP 100a, incl. H and bio CO2'),
     ('IPCC 2021 - dpCFsSSP119_MY2030 - year100', 'climate change', 'pGWP100'),
     ('IPCC 2021 - dpCFsSSP245_MY2030 - year100', 'climate change', 'pGWP100'),
     ('IPCC 2021 - dpCFsSSP585_MY2030 - year100', 'climate change', 'pGWP100'),
     ('IPCC 2021 - dpCFsSSP119_MY2040 - year100', 'climate change', 'pGWP100'),
     ('IPCC 2021 - dpCFsSSP245_MY2040 - year100', 'climate change', 'pGWP100'),
     ('IPCC 2021 - dpCFsSSP585_MY2040 - year100', 'climate change', 'pGWP100'),
     ('IPCC 2021 - dpCFsSSP119_MY2050 - year100', 'climate change', 'pGWP100'),
     ('IPCC 2021 - dpCFsSSP245_MY2050 - year100', 'climate change', 'pGWP100'),
     ('IPCC 2021 - dpCFsSSP585_MY2050 - year100', 'climate change', 'pGWP100')
]

In [ ]:

def calculate_everything(activities, method_list):
    results = np.zeros((len(activities), len(method_list)))
    
    # Use only the .key from the first activity
    lca = bw2calc.LCA({activities[0][0]: 1}, method=method_list[0])  
    lca.lci()
    lca.decompose_technosphere()
    lca.lcia()

    #builds characterization_matrices in the same order as gwp_all, so later we can append col_names in final DF 
    characterization_matrices = []
    for method in method_list:
        lca.switch_method(method)
        characterization_matrices.append(lca.characterization_matrix.copy())
        
    for first, activity in enumerate(activities):
        activity_key = activity[0]  # Extract just the key
        activity_name = activity[1] 
        activity_rp = activity[2] 
        lca.redo_lci({activity_key: 1})
        
        for second, matrix in enumerate(characterization_matrices):
            results[first, second] = (matrix * lca.inventory).sum()
            
    return results


### to prepare all act to be calculated 

In [ ]:
activities = sorted([
    (e.key, e['name'], e['reference product'])
    for a in agri
    for e in eidb
    if e['name'] == a['name'] 
    and e['location'] == a['location'] 
    and e['reference product'] == a['reference product']
])


In [ ]:
print(activities[36])

In [ ]:
len(activities)

In [ ]:
from time import time

start = time()
lca_scores = calculate_everything(activities, gwp_all)
elapsed = time() - start

In [ ]:
print(f"calculate all filtered process N = {len(agri)},  in {elapsed:.2f} seconds, at {lca_scores.shape[0] * lca_scores.shape[1] / elapsed:.1f} calculations per second")


In [ ]:
### 

In [ ]:
with open("calculated_lca_scores_N201.pkl", "wb") as f:
    pickle.dump(lca_scores, f)

In [ ]:
#with open("calculated_lca_scores_N201.pkl", "rb") as f:
#    lca_scores = pickle.load(f)

In [ ]:
print("Array shape:", lca_scores.shape)
#print("Number of activities:", len(activities))
#print("Average:", np.average(lca_scores))
#print("Median (of absolute value):", np.median(np.abs(lca_scores)))
#print("Min, Max:", np.min(lca_scores), np.max(lca_scores))
#print("Fraction of all scores with score of zero:", (lca_scores == 0).sum() / float(lca_scores.shape[0] * lca_scores.shape[1]))
#print("Methods where all scores are zero:", (lca_scores.sum(axis=0) == 0).sum())
#print("Activities where all scores are zero:", (lca_scores.sum(axis=1) == 0).sum())
#np.histogram(lca_scores)

In [ ]:
df = pd.DataFrame(lca_scores, columns=gwp_all)

df['name'] = [act[1] for act in activities]
df['reference product'] = [act[2] for act in activities]
df['activity key'] = [act[0] for act in activities]
df = df[[ 'name', 'reference product', 'activity key' ] + list(df.columns[:-3])]
df

### analysis & plot 

In [ ]:

def create_correlation_matrix(scores_array, method_names):
    num_methods = scores_array.shape[1]
    correlations = np.eye(num_methods)  # Start with identity matrix (diagonal = 1)

    for row in range(num_methods):
        for col in range(row + 1, num_methods):  # Only upper triangle
            dataset_1 = scores_array[:, row]
            dataset_2 = scores_array[:, col]
            mask = (dataset_1 != 0) & (dataset_2 != 0)  # Exclude zero scores
            tau, _ = stats.kendalltau(dataset_1[mask], dataset_2[mask])
            corr = 0 if np.isnan(tau) else tau
            correlations[row, col] = correlations[col, row] = corr  # Symmetric

    return correlations, method_names


In [ ]:
correlation_matrix, method_labels = create_correlation_matrix(lca_scores, gwp_all)
# mask upper triangle only (keep diagonal = 1.0)
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool), k=1)
masked_corr = np.ma.array(correlation_matrix, mask=mask)

fig, ax = plt.subplots(figsize=(12, 12))
cmap = plt.cm.coolwarm_r  # Reversed so RED = LOW, BLUE = HIGH

c = ax.imshow(masked_corr, cmap=cmap, vmin=masked_corr.min(), vmax=1.0)  #vmin dynamically change 

# labels
tick_labels = [str(m) for m in method_labels]
ax.set_xticks(np.arange(len(tick_labels)))
ax.set_yticks(np.arange(len(tick_labels)))
ax.set_xticklabels(tick_labels, rotation=90, fontsize=10)
ax.set_yticklabels(tick_labels, fontsize=10)

cb = fig.colorbar(c, ax=ax, shrink=0.85)
cb.ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))

plt.tight_layout()
plt.savefig("correlation_matrix_pGWPvsGWP_onN201_eidb-agri.png", dpi=300, bbox_inches='tight')

plt.show()


### top impacted activities 

In [ ]:
def rank_activity_gwp_variation(df, top_n=10, method='perc'):
    """
    Get variation across GWP methods for each activity and return:
    1. DataFrame with added variation column
    2. Top N activities with highest variation
    Parameters:
    - df: DataFrame with metadata + GWP method columns
    - top_n: number of top varying activities to return
    - method: 'range', 'std', or 'perc'
    Returns:
    - df_with_variation: full DataFrame with new variation column
    - df_top_variation: top N activities sorted by variation
    """
    gwp_columns = df.columns[3:]  # assuming first 3 columns are metadata
    variation_col = f"variation_{method}"  # dynamically name the column


    if method == 'range':
        df[variation_col] = df[gwp_columns].max(axis=1) - df[gwp_columns].min(axis=1)
    elif method == 'std':
        df[variation_col] = df[gwp_columns].std(axis=1)
    elif method == 'perc': 
        #df['variation'] = (df[gwp_columns].max(axis=1) - df[gwp_columns].min(axis=1)) / df[gwp_columns].max(axis=1)
        max_val = df[gwp_columns].max(axis=1)
        min_val = df[gwp_columns].min(axis=1)
        raw_perc = (max_val - min_val) / max_val * 100
        df[variation_col] = raw_perc.round().astype('Int64')  # preserves NaN + int formatting 
        
    else:
        raise ValueError("method must be 'range' or 'std' or 'perc' ")

    df_top = df.sort_values(by=variation_col, ascending=False).head(top_n)
    
    return df, df_top


In [ ]:
df = pd.DataFrame(lca_scores, columns=gwp_all)
df['name'] = [act[1] for act in activities]
df['reference product'] = [act[2] for act in activities]
df['activity key'] = [act[0] for act in activities]
df = df[[ 'name', 'reference product', 'activity key' ] + list(df.columns[:-3])]

df2, df_top = rank_activity_gwp_variation (df, top_n=20, method = 'perc')

In [ ]:
df_top

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_top_variation_profiles(df_top, gwp_columns, title, filename, font_family="Times New Roman"):
    """
    Plot top activities with highest GWP method variation in subplots.

    Parameters:
    - df_top: DataFrame with top N activities (e.g., from rank_activity_gwp_variation)
    - gwp_columns: List of GWP method columns (as-is, even if they're tuples)
    - title: Suptitle for the entire figure
    - filename: Path to save the figure (e.g., 'plot.png')
    - font_family: Optional global font (default: Times New Roman)
    """
    #  global font
    plt.rcParams["font.family"] = font_family
    # layout setup
    top_n = df_top.shape[0]
    ncols = 4
    nrows = int(np.ceil(top_n / ncols))
    
    fig, axs = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), sharex=True)
    axs = axs.flatten()

    # GWP method names to string for plotting
    x_labels = [str(col) for col in gwp_columns]

    for i, (idx, row) in enumerate(df_top.iterrows()):
        ax = axs[i]
        y = row[gwp_columns].astype(float)  # ensure numeric

        # Plot line
        ax.plot(x_labels, y, marker='o', linestyle='-', color='orange')

        # Highlight min and max
        min_idx = y.idxmin()
        max_idx = y.idxmax()
        ax.plot(str(min_idx), y[min_idx], marker='v', color='blue', markersize=10)
        ax.plot(str(max_idx), y[max_idx], marker='^', color='red', markersize=10)

        # Annotate largest % change from first point
        percent_changes = ((y - y.iloc[0]) / abs(y.iloc[0])) * 100
        largest_change_idx = percent_changes.abs().idxmax()
        percent_value = percent_changes[largest_change_idx]

        ax.annotate(f"{percent_value:.1f}%", 
                    xy=(str(largest_change_idx), y[largest_change_idx]),
                    xytext=(0, 10), textcoords='offset points',
                    ha='center', fontsize=9, color='black', fontweight='bold')

        #  title per panel
        name = row['name']
        ref_prod = row['reference product']
        label = f"{name}\n{ref_prod}"
        ax.set_title(label, fontsize=11)
        ax.set_xticks(range(len(x_labels)))
        ax.set_xticklabels(x_labels, rotation=90, fontsize=10)

        ax.grid(True, axis='y', linestyle='--', alpha=0.5)
        ax.set_ylabel(r"(p)GWP100 [kg CO$_2$-eq]")

    # hide any unused subplots
    for j in range(i + 1, len(axs)):
        fig.delaxes(axs[j])

    plt.suptitle(title, fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust layout to accommodate title
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
df = pd.DataFrame(lca_scores, columns=gwp_all)
df['name'] = [act[1] for act in activities]
df['reference product'] = [act[2] for act in activities]
df['activity key'] = [act[0] for act in activities]
df = df[[ 'name', 'reference product', 'activity key' ] + list(df.columns[:-3])]

df2, df_top = rank_activity_gwp_variation (df, top_n=12, method = 'perc')

plot_top_variation_profiles(
    df_top=df_top, 
    gwp_columns=df.columns[3:-1],  # exclude metadata and new col 'variation'
    title="Top 12 agricultural activities (ecoinvent cutoff v3.9.1) with highest (p)GWP100 method variation",
    filename="top12_gwp_variation_lineplot_fromN201_eidb-agri.png"
)
